# 03b · BiLSTM sentiment model (trained on LLM silver labels)

**Goal.** Learn a fast, cheap sentiment model for financial text that we can run on *every* headline and *every* Reddit comment.

**Teacher labels (no manual labeling).**
* FNSPID headlines + lead (8,784, from 03a): Qwen2.5-7B **calibrated** probabilities (p_bear, p_neu, p_bull), used as *soft targets*, weighted by LLM confidence.
* Twitter Financial News Sentiment **train** split (9,543, human labels), one-hot targets.

**Time-based split (no look-ahead).**

| split | data | use |
|---|---|---|
| train | FNSPID 2009–2020 + Twitter train | fit |
| val | FNSPID 2021 (meme year) | early stopping, model choice |
| test | FNSPID 2022–2023 | out-of-time test vs LLM labels |
| test (human) | Twitter validation split (2,388) | accuracy vs human labels |
| test (meme social) | 2,555 r/wallstreetbets BB comments scored by ChatGPT and Claude (`Comments_w_Labels`) | does it transfer to Reddit? |

**Model.** word2vec embeddings (trained on FNSPID ≤2021 + Twitter train + Reddit 2021, unsupervised) → BiLSTM(64) → mean+max pooling → dropout → linear(3). Three seeds; ensemble = average of probabilities.

**Outputs (DataSource/).** `bilstm_scores_fnspid.csv`, `bilstm_scores_reddit.csv`, `bilstm_metrics.csv`, `bilstm_sentiment.pt`.

In [1]:
# 0. Setup
!pip -q install "gensim>=4.4"
from google.colab import drive
drive.mount('/content/drive')

import os, glob, re, json, time, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, classification_report
from scipy import stats

def find_project_dir(name="dl-quant_project"):
    cands = [f"/content/drive/MyDrive/Colab Notebooks/{name}", f"/content/drive/MyDrive/{name}",
             *glob.glob(f"/content/drive/MyDrive/*/{name}"), *glob.glob(f"/content/drive/Shareddrives/*/{name}")]
    for p in cands:
        if os.path.isdir(p): return p
    raise FileNotFoundError("Add a shortcut to the shared 'dl-quant_project' folder in My Drive (see 00_START_HERE).")

PROJECT_DIR = find_project_dir()
DATA_DIR = os.path.join(PROJECT_DIR, "DataSource")
SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count())
LABELS = ["bearish", "neutral", "bullish"]          # class order used everywhere in this notebook
L2I = {l: i for i, l in enumerate(LABELS)}
print("Data folder:", DATA_DIR, "| device:", DEVICE)

MessageError: Error: credential propagation was unsuccessful

## 1. Load labeled data

In [ ]:
from datasets import load_dataset

L = pd.read_csv(os.path.join(DATA_DIR, "llm_labels_fnspid.csv"))
L["year"] = pd.to_datetime(L["date"]).dt.year
L["split"] = np.select([L.year <= 2020, L.year == 2021], ["train", "val"], "test")
L["text"] = L["input_text"].fillna(L["headline"]).astype(str)

tw = load_dataset("zeroshot/twitter-financial-news-sentiment")
TW_MAP = {0: "bearish", 1: "bullish", 2: "neutral"}          # dataset ids -> names
tw_train = tw["train"].to_pandas();      tw_train["label_name"] = tw_train["label"].map(TW_MAP)
tw_test  = tw["validation"].to_pandas(); tw_test["label_name"]  = tw_test["label"].map(TW_MAP)

def onehot(names):
    Q = np.zeros((len(names), 3), dtype=np.float32)
    Q[np.arange(len(names)), [L2I[n] for n in names]] = 1.0
    return Q

# training table: soft targets q, sample weight w, hard label y
fn_q = L[["p_bear", "p_neu", "p_bull"]].to_numpy(np.float32)
fn = pd.DataFrame({"text": L.text, "split": L.split, "source": "fnspid",
                   "y": L.llm_label.map(L2I), "w": L.llm_confidence.astype(np.float32)})
twd = pd.DataFrame({"text": tw_train.text, "split": "train", "source": "twitter",
                    "y": tw_train.label_name.map(L2I), "w": 1.0})
data = pd.concat([fn, twd], ignore_index=True)
Q_all = np.vstack([fn_q, onehot(tw_train.label_name)])

print(data.groupby(["split", "source"]).size().unstack(fill_value=0))
print("\nLLM label mix by split:\n", pd.crosstab(L.split, L.llm_label, normalize="index").round(2))

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

sent_train.csv:   0%|          | 0.00/859k [00:00<?, ?B/s]

sent_valid.csv:   0%|          | 0.00/217k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

source  fnspid  twitter
split                  
test      1658        0
train     4597     9543
val       2529        0

LLM label mix by split:
 llm_label  bearish  bullish  neutral
split                               
test          0.15     0.18     0.67
train         0.20     0.18     0.62
val           0.12     0.19     0.70


## 2. Tokenizer and word2vec embeddings
word2vec is trained only on text up to 2021 (FNSPID headlines + bodies, Twitter train, Reddit 2021 comments). No labels are used, and nothing from the 2022–23 test period is seen.

In [ ]:
URL = re.compile(r"https?://\S+|www\.\S+")
TOK = re.compile(r"\$[a-z]+|[a-z]+(?:'[a-z]+)?|\d+(?:[.,]\d+)?%?|[%!?]")
def tokenize(s):
    s = URL.sub(" ", str(s).lower())
    out = []
    for t in TOK.findall(s):
        if t.startswith("$"): out.append("<cashtag>")
        elif t[0].isdigit(): out.append("<pct>" if t.endswith("%") else "<num>")
        else: out.append(t)
    return out

print(tokenize("$GME soars 25% after RC joins board! https://t.co/x — shorts are squeezed"))

# Reddit comments (all files, de-duplicated by comment id); ticker from file name
reddit_files = sorted(glob.glob(os.path.join(DATA_DIR, "wallstreetbets_*.csv")))
parts = []
for f in reddit_files:
    d = pd.read_csv(f, usecols=["id", "body", "created_utc", "score"])
    d["ticker"] = os.path.basename(f).split("_")[1]
    parts.append(d)
R = pd.concat(parts, ignore_index=True).drop_duplicates(["ticker", "id"])
R = R[R.body.notna() & ~R.body.isin(["[deleted]", "[removed]"])].reset_index(drop=True)
print("Reddit comments:", len(R), R.ticker.value_counts().to_dict())

clean = pd.read_csv(os.path.join(DATA_DIR, "fnspid_meme_clean.csv"))
clean = clean[pd.to_datetime(clean.date).dt.year <= 2021]
corpus_text = (list(clean.Article_title.fillna("") + " . " + clean.text.fillna("").str.slice(0, 5000))
               + list(tw_train.text)
               + list(R.body.sample(min(len(R), 300_000), random_state=SEED)))
t0 = time.time()
corpus = [tokenize(s) for s in corpus_text]
print(f"word2vec corpus: {len(corpus):,} docs, {sum(map(len, corpus)):,} tokens ({time.time()-t0:.0f}s to tokenize)")

['<cashtag>', 'soars', '<pct>', 'after', 'rc', 'joins', 'board', '!', 'shorts', 'are', 'squeezed']
Reddit comments: 912984 {'GME': 440228, 'AMC': 296687, 'BB': 176069}
word2vec corpus: 317,249 docs, 10,284,948 tokens (7s to tokenize)


In [ ]:
from gensim.models import Word2Vec
t0 = time.time()
w2v = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=3, sg=1, negative=5,
               epochs=5, workers=os.cpu_count(), seed=SEED)
print(f"word2vec trained in {time.time()-t0:.0f}s; vocab {len(w2v.wv):,}")
for w in ["bullish", "squeeze", "downgrade", "bankruptcy", "moon"]:
    if w in w2v.wv: print(w, "->", [x for x, _ in w2v.wv.most_similar(w, topn=6)])

word2vec trained in 164s; vocab 30,922
bullish -> ['bearish', 'optimistic', 'pennant', 'bull', 'engulfing', 'overbought']
squeeze -> ['sqeeze', 'squeezes', 'gamma', 'squueze', 'squezze', 'sqoze']
downgrade -> ['underperform', 'wold', 'handler', 'jaffray', 'jefferies', 'walkley']
bankruptcy -> ['stave', 'homerun', 'staved', 'brink', 'bankrupt', 'bankrupting']
moon -> ['moooooon', 'mars', 'mooooon', 'mooon', 'moooon', 'pluto']


## 3. Vocabulary, tensors and the BiLSTM

In [ ]:
MAX_VOCAB, MAX_LEN, EMB = 40_000, 64, 100
itos = ["<pad>", "<unk>"] + w2v.wv.index_to_key[:MAX_VOCAB]
stoi = {w: i for i, w in enumerate(itos)}
rng = np.random.default_rng(SEED)
emb_matrix = np.zeros((len(itos), EMB), dtype=np.float32)
emb_matrix[1] = rng.normal(0, 0.1, EMB)
for w, i in stoi.items():
    if i >= 2: emb_matrix[i] = w2v.wv[w]

def encode(texts, max_len=MAX_LEN):
    X = np.zeros((len(texts), max_len), dtype=np.int64); lens = np.ones(len(texts), dtype=np.int64)
    for r, s in enumerate(texts):
        ids = [stoi.get(t, 1) for t in tokenize(s)][:max_len] or [1]
        X[r, :len(ids)] = ids; lens[r] = len(ids)
    return torch.from_numpy(X), torch.from_numpy(lens)

X_all, len_all = encode(data.text.tolist())
cover = (X_all[X_all > 0] != 1).float().mean().item()
print(f"encoded {len(data):,} texts | token coverage by vocab {cover:.1%} | truncated at {MAX_LEN}: {(len_all == MAX_LEN).float().mean().item():.1%}")

class BiLSTMClf(nn.Module):
    def __init__(self, emb, hidden=64, n_cls=3, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.tensor(emb), freeze=False, padding_idx=0)
        self.emb_drop = nn.Dropout(0.2)
        self.lstm = nn.LSTM(emb.shape[1], hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(4 * hidden, n_cls)
    def forward(self, x, lens):
        e = self.emb_drop(self.emb(x))
        packed = nn.utils.rnn.pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        mask = (x != 0).unsqueeze(-1)
        mean = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        mx = out.masked_fill(~mask, -1e4).max(1).values
        return self.fc(self.drop(torch.cat([mean, mx], 1)))

print(BiLSTMClf(emb_matrix))
print("trainable parameters:", sum(p.numel() for p in BiLSTMClf(emb_matrix).parameters()))

encoded 18,327 texts | token coverage by vocab 98.5% | truncated at 64: 9.2%
BiLSTMClf(
  (emb): Embedding(30924, 100, padding_idx=0)
  (emb_drop): Dropout(p=0.2, inplace=False)
  (lstm): LSTM(100, 64, batch_first=True, bidirectional=True)
  (drop): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)
trainable parameters: 3178163


## 4. Training (soft-target cross-entropy, class weights, early stopping on 2021 macro-F1)

In [ ]:
tr = np.where(data.split == "train")[0]; va = np.where(data.split == "val")[0]; te = np.where(data.split == "test")[0]
y_all = data.y.to_numpy(); w_all = data.w.to_numpy(np.float32)
counts = np.bincount(y_all[tr], minlength=3)
class_w = torch.tensor(len(tr) / (3 * counts), dtype=torch.float32)
print("train class counts", dict(zip(LABELS, counts)), "-> class weights", class_w.numpy().round(2))

Q_t = torch.tensor(Q_all); W_t = torch.tensor(w_all); Y_t = torch.tensor(y_all)

@torch.no_grad()
def predict_proba(model, X, lens, bs=1024):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(F.softmax(model(X[i:i+bs].to(DEVICE), lens[i:i+bs].to(DEVICE)), -1).cpu())
    return torch.cat(out).numpy()

def train_one(seed, epochs=15, patience=3, bs=64, lr=2e-3):
    set_seed(seed)
    model = BiLSTMClf(emb_matrix).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    cw = class_w.to(DEVICE)
    best, best_state, bad, hist = -1, None, 0, []
    for ep in range(1, epochs + 1):
        model.train(); perm = np.random.permutation(tr); tot = 0; t0 = time.time()
        for i in range(0, len(perm), bs):
            b = torch.from_numpy(perm[i:i+bs])
            logits = model(X_all[b].to(DEVICE), len_all[b].to(DEVICE))
            q = Q_t[b].to(DEVICE)
            loss_i = -(q * F.log_softmax(logits, -1)).sum(1) * cw[Y_t[b].to(DEVICE)] * W_t[b].to(DEVICE)
            loss = loss_i.mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item() * len(b)
        pv = predict_proba(model, X_all[va], len_all[va])
        f1 = f1_score(y_all[va], pv.argmax(1), average="macro")
        hist.append((ep, tot / len(tr), f1))
        print(f"  seed {seed} epoch {ep:2d} | train loss {tot/len(tr):.4f} | val(2021) macro-F1 {f1:.3f} | {time.time()-t0:.0f}s")
        if f1 > best: best, bad, best_state = f1, 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model, best, hist

SEEDS = [42, 43, 44]
models, histories = [], {}
for s in SEEDS:
    m, best, hist = train_one(s); models.append(m); histories[s] = hist
    print(f"seed {s}: best val macro-F1 {best:.3f} at epoch {max(hist, key=lambda h: h[2])[0]}")

train class counts {'bearish': np.int64(2377), 'neutral': np.int64(9008), 'bullish': np.int64(2755)} -> class weights [1.98 0.52 1.71]
  seed 42 epoch  1 | train loss 0.7835 | val(2021) macro-F1 0.462 | 32s
  seed 42 epoch  2 | train loss 0.5187 | val(2021) macro-F1 0.539 | 35s
  seed 42 epoch  3 | train loss 0.3953 | val(2021) macro-F1 0.585 | 32s
  seed 42 epoch  4 | train loss 0.3126 | val(2021) macro-F1 0.600 | 30s
  seed 42 epoch  5 | train loss 0.2611 | val(2021) macro-F1 0.563 | 32s
  seed 42 epoch  6 | train loss 0.2340 | val(2021) macro-F1 0.543 | 29s
  seed 42 epoch  7 | train loss 0.2132 | val(2021) macro-F1 0.585 | 31s
seed 42: best val macro-F1 0.600 at epoch 4
  seed 43 epoch  1 | train loss 0.7744 | val(2021) macro-F1 0.482 | 30s
  seed 43 epoch  2 | train loss 0.5225 | val(2021) macro-F1 0.551 | 31s
  seed 43 epoch  3 | train loss 0.3959 | val(2021) macro-F1 0.514 | 30s
  seed 43 epoch  4 | train loss 0.3176 | val(2021) macro-F1 0.553 | 30s
  seed 43 epoch  5 | train lo

## 5. Evaluation

In [ ]:
def ens_proba(texts):
    X, ln = encode(list(texts))
    return np.mean([predict_proba(m, X, ln) for m in models], axis=0)

def metrics(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_F1": f1_score(y_true, y_pred, average="macro"),
            "kappa": cohen_kappa_score(y_true, y_pred)}

rows = []
P_fn = ens_proba(L.text)                       # all FNSPID headlines
L["b_pbear"], L["b_pneu"], L["b_pbull"] = P_fn[:, 0], P_fn[:, 1], P_fn[:, 2]
L["bilstm_label"] = [LABELS[i] for i in P_fn.argmax(1)]
yL = L.llm_label.map(L2I).to_numpy()
for sp in ["val", "test"]:
    idx = (L.split == sp).to_numpy()
    name = "FNSPID 2021 (val) vs LLM labels" if sp == "val" else "FNSPID 2022-23 (test) vs LLM labels"
    rows.append({"eval_set": name, "model": "BiLSTM ensemble", "n": idx.sum(), **metrics(yL[idx], P_fn[idx].argmax(1))})
    X, ln = encode(L.text[idx].tolist())
    for s, m in zip(SEEDS, models):
        rows.append({"eval_set": name, "model": f"BiLSTM seed {s}", "n": idx.sum(), **metrics(yL[idx], predict_proba(m, X, ln).argmax(1))})

P_tw = ens_proba(tw_test.text)
y_tw = tw_test.label_name.map(L2I).to_numpy()
rows.append({"eval_set": "Twitter test vs HUMAN labels", "model": "BiLSTM ensemble", "n": len(y_tw), **metrics(y_tw, P_tw.argmax(1))})
X, ln = encode(tw_test.text.tolist())
for s, m in zip(SEEDS, models):
    rows.append({"eval_set": "Twitter test vs HUMAN labels", "model": f"BiLSTM seed {s}", "n": len(y_tw), **metrics(y_tw, predict_proba(m, X, ln).argmax(1))})

# references: the LLM teacher on the same Twitter texts (03a) and the keyword baseline (02)
chk = pd.read_csv(os.path.join(DATA_DIR, "llm_check_twitter.csv"))
rows.append({"eval_set": "Twitter test vs HUMAN labels", "model": "LLM teacher (Qwen2.5-7B, calibrated)", "n": len(chk),
             **metrics(chk["true"].map(L2I), chk["llm_cal"].map(L2I))})
rows.append({"eval_set": "Twitter test vs HUMAN labels", "model": "Keyword LM + market words (02)", "n": 2388,
             "accuracy": 0.621, "macro_F1": 0.504, "kappa": np.nan})
rows.append({"eval_set": "Twitter test vs HUMAN labels", "model": "Majority class (neutral)", "n": len(y_tw),
             **metrics(y_tw, np.full(len(y_tw), L2I["neutral"]))})

# TF-IDF + logistic regression baseline trained on the same data and labels
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
tf = TfidfVectorizer(tokenizer=tokenize, lowercase=False, ngram_range=(1, 2), min_df=2, token_pattern=None)
Xtr = tf.fit_transform(data.text.iloc[tr])
lr_clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=2.0).fit(Xtr, y_all[tr], sample_weight=w_all[tr])
for name, texts, ytrue in [("FNSPID 2021 (val) vs LLM labels", L.text[L.split == "val"], yL[(L.split == "val").to_numpy()]),
                           ("FNSPID 2022-23 (test) vs LLM labels", L.text[L.split == "test"], yL[(L.split == "test").to_numpy()]),
                           ("Twitter test vs HUMAN labels", tw_test.text, y_tw)]:
    rows.append({"eval_set": name, "model": "TF-IDF + logistic regression", "n": len(ytrue), **metrics(ytrue, lr_clf.predict(tf.transform(texts)))})

M = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(M.round(3).to_string(index=False))
seed_rows = M[M.model.str.startswith("BiLSTM seed")]
print("\nBiLSTM per-seed mean ± sd:\n", seed_rows.groupby("eval_set")[["accuracy", "macro_F1"]].agg(["mean", "std"]).round(3))

                           eval_set                                model    n  accuracy  macro_F1  kappa
    FNSPID 2021 (val) vs LLM labels                      BiLSTM ensemble 2529     0.702     0.607  0.404
    FNSPID 2021 (val) vs LLM labels                       BiLSTM seed 42 2529     0.703     0.600  0.397
    FNSPID 2021 (val) vs LLM labels                       BiLSTM seed 43 2529     0.669     0.590  0.372
    FNSPID 2021 (val) vs LLM labels                       BiLSTM seed 44 2529     0.693     0.588  0.377
FNSPID 2022-23 (test) vs LLM labels                      BiLSTM ensemble 1658     0.747     0.652  0.485
FNSPID 2022-23 (test) vs LLM labels                       BiLSTM seed 42 1658     0.744     0.648  0.481
FNSPID 2022-23 (test) vs LLM labels                       BiLSTM seed 43 1658     0.720     0.637  0.456
FNSPID 2022-23 (test) vs LLM labels                       BiLSTM seed 44 1658     0.722     0.622  0.438
       Twitter test vs HUMAN labels                    

In [1]:
print("Twitter test (human labels), BiLSTM ensemble:\n")
print(classification_report(y_tw, P_tw.argmax(1), target_names=LABELS, digits=3))
print("confusion (rows = human, cols = BiLSTM):\n", pd.DataFrame(confusion_matrix(y_tw, P_tw.argmax(1)), index=LABELS, columns=LABELS))
idx = (L.split == "test").to_numpy()
print("\nFNSPID 2022-23 confusion (rows = LLM, cols = BiLSTM):\n",
      pd.DataFrame(confusion_matrix(yL[idx], P_fn[idx].argmax(1)), index=LABELS, columns=LABELS))

KeyboardInterrupt: 

### 5b. Price-consistency check (out of sample)
Same-day log return on the article date: is it higher for headlines the model calls bullish than for bearish ones?

In [ ]:
px = pd.read_csv(os.path.join(DATA_DIR, "prices_gme_amc_bb.csv"), header=[0, 1], index_col=0, skiprows=[2])
px.index = pd.to_datetime(px.index)
adj = px["Adj Close"]
ret = np.log(adj).diff().stack().rename("ret").reset_index()
ret.columns = ["date", "ticker", "ret"]
Lr = L.assign(date=pd.to_datetime(L.date)).merge(ret, on=["date", "ticker"], how="inner")

def spread(df, col):
    b, s = df.loc[df[col] == "bullish", "ret"], df.loc[df[col] == "bearish", "ret"]
    t, p = stats.ttest_ind(b, s, equal_var=False)
    return pd.Series({"n_bull": len(b), "n_bear": len(s), "spread_pp": 100 * (b.mean() - s.mean()), "t": t, "p": p})

pc = []
for period, mask in [("2021 (val)", Lr.year == 2021), ("2022-23 (test)", Lr.year >= 2022), ("2009-20 (train, in-sample)", Lr.year <= 2020)]:
    for col, name in [("bilstm_label", "BiLSTM"), ("llm_label", "LLM teacher")]:
        pc.append({"period": period, "labels": name, **spread(Lr[mask], col)})
PC = pd.DataFrame(pc)
print(PC.round(3).to_string(index=False))

                    period      labels  n_bull  n_bear  spread_pp      t   p
                2021 (val)      BiLSTM   472.0   398.0      8.267  4.463 0.0
                2021 (val) LLM teacher   450.0   273.0     17.079  8.525 0.0
            2022-23 (test)      BiLSTM   207.0   300.0      4.935  5.530 0.0
            2022-23 (test) LLM teacher   282.0   239.0      5.019  5.787 0.0
2009-20 (train, in-sample)      BiLSTM   801.0   973.0      6.247 16.885 0.0
2009-20 (train, in-sample) LLM teacher   789.0   891.0      6.415 16.797 0.0


### 5c. Transfer to meme-stock Reddit comments
`Comments_w_Labels`: 2,555 r/wallstreetbets comments on BB (30 Nov – 8 Dec 2020), each scored from −1 to +1 by ChatGPT and by Claude (a teammate's LLM labels). The BiLSTM never saw them. We compare its score p_bull − p_bear with both.

In [ ]:
cd = os.path.join(DATA_DIR, "Comments_w_Labels")
gpt = pd.read_csv(os.path.join(cd, "wallstreetbets_BB_Nov30_2020_Dec9_2020_body_Only_w_ChatGPTLabels.csv"), header=None, names=["body", "gpt"])
cla = pd.read_csv(os.path.join(cd, "wallstreetbets_BB_Nov30_2020_Dec9_2020_body_Only_w_ClaudeLabels.csv"), header=None, names=["body", "claude"])
C = gpt.copy(); C["claude"] = cla["claude"].values
C = C[pd.to_numeric(C.gpt, errors="coerce").notna() & pd.to_numeric(C.claude, errors="coerce").notna()].astype({"gpt": float, "claude": float})
Pc = ens_proba(C.body)
C["bilstm"] = Pc[:, 2] - Pc[:, 0]
def sgn(x, thr=0.15): return np.where(x > thr, 1, np.where(x < -thr, -1, 0))
rc = []
for a, b in [("bilstm", "gpt"), ("bilstm", "claude"), ("gpt", "claude")]:
    rho, p = stats.spearmanr(C[a], C[b])
    rc.append({"pair": f"{a} vs {b}", "n": len(C), "spearman": rho, "p": p,
               "3-class agreement": (sgn(C[a]) == sgn(C[b])).mean(), "kappa": cohen_kappa_score(sgn(C[a]), sgn(C[b]))})
RC = pd.DataFrame(rc); print(RC.round(3).to_string(index=False))
print("\nmean score: BiLSTM %.3f | ChatGPT %.3f | Claude %.3f" % (C.bilstm.mean(), C.gpt.mean(), C.claude.mean()))

            pair    n  spearman   p  3-class agreement  kappa
   bilstm vs gpt 2556     0.105 0.0              0.428 -0.023
bilstm vs claude 2556     0.309 0.0              0.441  0.140
   gpt vs claude 2556     0.111 0.0              0.338  0.054

mean score: BiLSTM 0.111 | ChatGPT 0.028 | Claude 0.223


## 6. Score all Reddit comments and save everything

In [ ]:
t0 = time.time()
P_r = np.zeros((len(R), 3), dtype=np.float32)
CH = 50_000
for i in range(0, len(R), CH):
    P_r[i:i+CH] = ens_proba(R.body.iloc[i:i+CH])
    print(f"  scored {min(i+CH, len(R)):,}/{len(R):,} ({time.time()-t0:.0f}s)")
R_out = R[["ticker", "id", "created_utc", "score"]].copy()
R_out["p_bear"], R_out["p_neu"], R_out["p_bull"] = P_r[:, 0], P_r[:, 1], P_r[:, 2]
R_out["n_tokens"] = R.body.str.split().str.len()
R_out.to_csv(os.path.join(DATA_DIR, "bilstm_scores_reddit.csv"), index=False)

F_out = L[["ticker", "date", "target_date", "headline", "Url", "split", "llm_label", "p_bear", "p_neu", "p_bull",
           "bilstm_label", "b_pbear", "b_pneu", "b_pbull"]]
F_out.to_csv(os.path.join(DATA_DIR, "bilstm_scores_fnspid.csv"), index=False)

M.to_csv(os.path.join(DATA_DIR, "bilstm_metrics.csv"), index=False)
PC.to_csv(os.path.join(DATA_DIR, "bilstm_price_check.csv"), index=False)
RC.to_csv(os.path.join(DATA_DIR, "bilstm_reddit_check.csv"), index=False)
torch.save({"states": [m.state_dict() for m in models], "itos": itos, "max_len": MAX_LEN, "labels": LABELS,
            "config": {"emb": EMB, "hidden": 64}}, os.path.join(DATA_DIR, "bilstm_sentiment.pt"))
print("Reddit label mix:", pd.Series(P_r.argmax(1)).map(dict(enumerate(LABELS))).value_counts(normalize=True).round(3).to_dict())
print("saved:", [f for f in os.listdir(DATA_DIR) if f.startswith("bilstm")])

  scored 50,000/912,984 (33s)
  scored 100,000/912,984 (68s)
  scored 150,000/912,984 (103s)
  scored 200,000/912,984 (139s)
  scored 250,000/912,984 (171s)
  scored 300,000/912,984 (202s)
  scored 350,000/912,984 (229s)
  scored 400,000/912,984 (262s)
  scored 450,000/912,984 (290s)
  scored 500,000/912,984 (332s)
  scored 550,000/912,984 (372s)
  scored 600,000/912,984 (410s)
  scored 650,000/912,984 (450s)
  scored 700,000/912,984 (496s)
  scored 750,000/912,984 (540s)
  scored 800,000/912,984 (586s)
  scored 850,000/912,984 (621s)
  scored 900,000/912,984 (657s)
  scored 912,984/912,984 (665s)
Reddit label mix: {'neutral': 0.793, 'bullish': 0.113, 'bearish': 0.094}
saved: ['bilstm_scores_reddit.csv', 'bilstm_scores_fnspid.csv', 'bilstm_metrics.csv', 'bilstm_price_check.csv', 'bilstm_reddit_check.csv', 'bilstm_sentiment.pt']
